# SPX VIX-Style Term Structure — ThetaData v3, v0.6 Friday-Cycle Holiday Adjusted

Version: v0.6  
Data source: ThetaData v3 terminal  

Purpose: build a VIX-style SPX implied volatility term structure for custom tenors.  

Methodology upgrade from v0.5: use a Friday-cycle expiration universe rather than literal Friday-only expirations. This includes Thursday expirations only when the following Friday is not an SPX trading day.


In [1]:
# ============================================================
# Imports and global settings
# ============================================================

import io
import math
import requests
import numpy as np
import pandas as pd

from datetime import date, datetime


# ThetaData v3 local terminal endpoint
BASE_URL_V3 = "http://127.0.0.1:25503/v3"


# VIX calculation time: 4:00 PM ET
# Milliseconds after midnight
CALC_TIME_MS = 16 * 60 * 60 * 1000


# Temporary flat risk-free rate.
# We will improve this later with ThetaData rates.
DEFAULT_RISK_FREE_RATE = 0.05


# VIX methodology constants
MINUTES_PER_DAY = 1440
MINUTES_PER_YEAR = 525600
MINUTES_30_DAYS = 43200

# Target tenors for first custom term-structure build
TARGET_TENOR_DAYS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

# Keep 30d as the primary validation tenor because we can compare it to official VIX
PRIMARY_VALIDATION_TENOR_DAYS = 30

In [2]:
# ============================================================
# Date and time helper functions
# ============================================================

def yyyymmdd_to_date(x):
    """
    Convert 20240116 to datetime.date(2024, 1, 16).
    """
    s = str(int(x))
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))


def yyyymmdd_to_dash_string(x):
    """
    Convert 20240116 to '2024-01-16'.
    """
    dt = yyyymmdd_to_date(x)
    return dt.strftime("%Y-%m-%d")


def ms_to_time_string(ms_of_day):
    """
    Convert milliseconds after midnight to ThetaData v3 time string.

    Example:
    57600000 -> '16:00:00.000'
    """
    total_seconds = int(ms_of_day // 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}.000"


def is_third_friday(dt):
    """
    True if date is the standard monthly SPX expiration Friday.
    """
    return dt.weekday() == 4 and 15 <= dt.day <= 21

In [3]:
# ============================================================
# ThetaData v3 expiration loader
# ============================================================

def list_expirations_v3(symbol):
    """
    Get option expirations from ThetaData v3.

    v3 endpoint returns CSV text like:
        symbol,expiration
        "SPX","2024-02-16"

    This function converts expiration dates to YYYYMMDD integers.
    """
    url = BASE_URL_V3 + "/option/list/expirations"

    params = {
        "symbol": symbol
    }

    r = requests.get(url, params=params, timeout=60)

    print("URL:", r.url)
    print("Status code:", r.status_code)

    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))

    if df.empty:
        raise ValueError(f"No expirations returned for {symbol}")

    if "expiration" not in df.columns:
        raise ValueError(f"Could not find 'expiration' column. Columns are: {list(df.columns)}")

    expirations = (
        pd.to_datetime(df["expiration"])
        .dt.strftime("%Y%m%d")
        .astype(int)
        .tolist()
    )

    return sorted(expirations)


spx_exps = list_expirations_v3("SPX")
spxw_exps = list_expirations_v3("SPXW")

print()
print("SPX expirations loaded:", len(spx_exps))
print("SPXW expirations loaded:", len(spxw_exps))

print("SPX sample:", spx_exps[:5], "...", spx_exps[-5:])
print("SPXW sample:", spxw_exps[:5], "...", spxw_exps[-5:])

URL: http://127.0.0.1:25503/v3/option/list/expirations?symbol=SPX
Status code: 200
URL: http://127.0.0.1:25503/v3/option/list/expirations?symbol=SPXW
Status code: 200

SPX expirations loaded: 203
SPXW expirations loaded: 2193
SPX sample: [20120616, 20120721, 20120818, 20120922, 20121020] ... [20890221, 20890718, 20891219, 20900116, 20900618]
SPXW sample: [20120601, 20120608, 20120622, 20120706, 20120713] ... [20880920, 20880930, 20881018, 20890331, 20890630]


In [4]:
# ============================================================
# Rate settings
# ============================================================

DEFAULT_RATE_SYMBOL = "SOFR"

In [5]:
# ============================================================
# ThetaData v3 interest-rate loader
# ============================================================

def get_interest_rate_history_eod_v3(symbol, start_date, end_date, verbose=False):
    """
    Pull end-of-day interest-rate history from ThetaData v3.

    Example endpoint:
        /interest_rate/history/eod?symbol=SOFR&start_date=20240116&end_date=20240119

    ThetaData returns rates in percent form:
        5.3200 means 5.32%

    This function converts that to decimal form:
        0.0532
    """
    url = BASE_URL_V3 + "/interest_rate/history/eod"

    params = {
        "symbol": symbol,
        "start_date": int(start_date),
        "end_date": int(end_date),
        "format": "json"
    }

    r = requests.get(url, params=params, timeout=60)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)
        print("Response start:")
        print(r.text[:1000])

    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError(f"No interest-rate data returned for {symbol}")

    if "rate" not in df.columns or "created" not in df.columns:
        raise ValueError(f"Unexpected rate response columns: {list(df.columns)}")

    df["trade_date"] = pd.to_datetime(df["created"]).dt.strftime("%Y%m%d").astype(int)
    df["rate_pct"] = pd.to_numeric(df["rate"], errors="coerce")
    df["rate_decimal"] = df["rate_pct"] / 100.0
    df["symbol"] = symbol

    return df[["symbol", "trade_date", "created", "rate_pct", "rate_decimal"]].copy()


def get_interest_rate_for_date_v3(symbol, trade_date, verbose=False):
    """
    Pull one same-date interest rate and return it in decimal form.

    Example:
        5.32% -> 0.0532
    """
    rate_df = get_interest_rate_history_eod_v3(
        symbol=symbol,
        start_date=trade_date,
        end_date=trade_date,
        verbose=verbose
    )

    matching = rate_df[rate_df["trade_date"] == int(trade_date)]

    if matching.empty:
        raise ValueError(f"No {symbol} rate found for {trade_date}")

    return float(matching.iloc[0]["rate_decimal"])

In [6]:
# ============================================================
# ThetaData v3 option-chain loader
# ============================================================

def get_chain_at_time(root, expi, trade_date, calc_time_ms, verbose=False):
    """
    Pull all strikes / both calls and puts for one SPX or SPXW expiration
    at a specific time using ThetaData v3.

    Returns dataframe with columns expected by the VIX math:
        root, expiration, strike, right, bid, ask, mid
    """

    request_time = ms_to_time_string(calc_time_ms)

    url = BASE_URL_V3 + "/option/history/quote"

    params = {
        "symbol": root,
        "expiration": yyyymmdd_to_dash_string(expi),
        "strike": "*",
        "right": "both",
        "start_date": yyyymmdd_to_dash_string(trade_date),
        "end_date": yyyymmdd_to_dash_string(trade_date),
        "start_time": request_time,
        "end_time": request_time,
        "interval": "1m",
        "format": "json"
    }

    r = requests.get(url, params=params, timeout=180)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)

    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError(
            f"No data returned for {root} {expi} on {trade_date} at {request_time}"
        )

    df["root"] = df["symbol"]
    df["expiration"] = pd.to_datetime(df["expiration"]).dt.strftime("%Y%m%d").astype(int)

    right_map = {
        "CALL": "C",
        "PUT": "P",
        "C": "C",
        "P": "P"
    }

    df["right"] = df["right"].map(right_map)

    if df["right"].isna().any():
        bad_values = df["right"].unique()
        raise ValueError(f"Unknown option right values found: {bad_values}")

    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")

    df["mid"] = (df["bid"] + df["ask"]) / 2

    keep_cols = [
        "root",
        "expiration",
        "strike",
        "right",
        "bid",
        "ask",
        "mid",
        "bid_size",
        "ask_size",
        "bid_exchange",
        "ask_exchange",
        "bid_condition",
        "ask_condition",
        "timestamp"
    ]

    df = df[keep_cols].copy()

    return df

In [7]:
# ============================================================
# Core VIX math functions
# ============================================================

def preferred_root_for_expiration(exp_yyyymmdd):
    """
    Prefer SPX for standard monthly third-Friday expirations.
    Otherwise use SPXW.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if is_third_friday(exp_dt) and exp_yyyymmdd in spx_exps:
        return "SPX"

    if exp_yyyymmdd in spxw_exps:
        return "SPXW"

    if exp_yyyymmdd in spx_exps:
        return "SPX"

    raise ValueError(f"Expiration {exp_yyyymmdd} not found in SPX or SPXW expiration lists")


def settlement_minutes_after_midnight_et(root):
    """
    Approximate VIX methodology settlement timing.

    SPX monthly options are AM-settled.
    SPXW weekly options are PM-settled.
    """
    if root == "SPX":
        return 9 * 60 + 30       # 9:30 AM ET

    if root == "SPXW":
        return 16 * 60           # 4:00 PM ET

    raise ValueError(f"Unknown root: {root}")


def minutes_to_expiry_vix_method(trade_date, exp_yyyymmdd, root, calc_time_ms):
    """
    Minutes from calculation time to option settlement time.
    """
    trade_dt = yyyymmdd_to_date(trade_date)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes_after_midnight = int(calc_time_ms // 60000)
    settlement_minutes = settlement_minutes_after_midnight_et(root)

    days_diff = (exp_dt - trade_dt).days

    return days_diff * MINUTES_PER_DAY + settlement_minutes - calc_minutes_after_midnight


def choose_vix_expirations_v3(trade_date, calc_time_ms=CALC_TIME_MS):
    """
    Select the two VIX-style expirations.

    Uses Friday expirations with roughly 23 to 37 days to expiry.
    Chooses the first two valid expirations.
    """
    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))

    candidates = []

    for exp in all_exps:
        exp_dt = yyyymmdd_to_date(exp)

        # Standard 30-day VIX uses Friday expirations.
        if exp_dt.weekday() != 4:
            continue

        root = preferred_root_for_expiration(exp)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms
        )

        days = minutes / MINUTES_PER_DAY

        if days > 23 and days < 37:
            candidates.append({
                "root": root,
                "expiration": exp,
                "minutes": minutes,
                "days": days
            })

    candidates = sorted(candidates, key=lambda x: x["minutes"])

    if len(candidates) < 2:
        raise ValueError(f"Could not find two valid VIX expirations for {trade_date}")

    return candidates[0], candidates[1]


def _prepare_call_put_tables(chain):
    """
    Create call and put tables indexed by strike.
    """
    df = chain.copy()

    df["strike"] = pd.to_numeric(df["strike"], errors="coerce")
    df["bid"] = pd.to_numeric(df["bid"], errors="coerce")
    df["ask"] = pd.to_numeric(df["ask"], errors="coerce")
    df["mid"] = pd.to_numeric(df["mid"], errors="coerce")

    df = df.dropna(subset=["strike", "bid", "ask", "mid", "right"])
    df = df[df["ask"] >= 0]
    df = df[df["bid"] >= 0]

    calls = (
        df[df["right"] == "C"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    puts = (
        df[df["right"] == "P"]
        .sort_values("strike")
        .drop_duplicates(subset=["strike"], keep="last")
        .set_index("strike")
    )

    return calls, puts


def _select_otm_options_with_bid_rule(options_df, ascending=True):
    """
    Select OTM options using a simplified Cboe-style nonzero-bid rule.

    Walk away from ATM.
    Keep options with positive bid.
    Stop after two consecutive zero-bid options.
    """
    options_df = options_df.sort_values("strike", ascending=ascending)

    selected_rows = []
    consecutive_zero_bids = 0

    for _, row in options_df.iterrows():
        if row["bid"] <= 0:
            consecutive_zero_bids += 1

            if consecutive_zero_bids >= 2:
                break

            continue

        consecutive_zero_bids = 0
        selected_rows.append(row)

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=options_df.columns)

    return pd.DataFrame(selected_rows)


def calc_single_term_variance(chain, minutes_to_expiry, r=DEFAULT_RISK_FREE_RATE):
    """
    Calculate VIX-style variance for one expiration.
    """
    T = minutes_to_expiry / MINUTES_PER_YEAR

    calls, puts = _prepare_call_put_tables(chain)

    common_strikes = sorted(set(calls.index).intersection(set(puts.index)))

    if len(common_strikes) == 0:
        raise ValueError("No common call/put strikes found")

    # Infer forward using the strike where call/put mids are closest.
    parity_rows = []

    for K in common_strikes:
        call_mid = calls.loc[K, "mid"]
        put_mid = puts.loc[K, "mid"]
        diff = abs(call_mid - put_mid)

        parity_rows.append({
            "strike": K,
            "call_mid": call_mid,
            "put_mid": put_mid,
            "abs_call_put_diff": diff
        })

    parity_df = pd.DataFrame(parity_rows)
    min_row = parity_df.loc[parity_df["abs_call_put_diff"].idxmin()]

    K_star = float(min_row["strike"])
    call_mid_star = float(min_row["call_mid"])
    put_mid_star = float(min_row["put_mid"])

    F = K_star + math.exp(r * T) * (call_mid_star - put_mid_star)

    all_strikes = sorted(set(calls.index).union(set(puts.index)))
    strikes_below_or_equal_forward = [K for K in all_strikes if K <= F]

    if len(strikes_below_or_equal_forward) == 0:
        raise ValueError("Could not find K0 below forward")

    K0 = max(strikes_below_or_equal_forward)

    # OTM puts below K0, moving downward from K0.
    put_rows = []

    for K in sorted([x for x in puts.index if x < K0], reverse=True):
        row = puts.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        put_rows.append(row)

    put_otm_raw = pd.DataFrame(put_rows)
    put_otm = _select_otm_options_with_bid_rule(put_otm_raw, ascending=False)

    # OTM calls above K0, moving upward from K0.
    call_rows = []

    for K in sorted([x for x in calls.index if x > K0]):
        row = calls.loc[K].copy()
        row["strike"] = K
        row["QK"] = row["mid"]
        call_rows.append(row)

    call_otm_raw = pd.DataFrame(call_rows)
    call_otm = _select_otm_options_with_bid_rule(call_otm_raw, ascending=True)

    # K0 option uses average of call and put mid.
    if K0 not in calls.index or K0 not in puts.index:
        raise ValueError(f"K0={K0} missing call or put quote")

    k0_row = calls.loc[K0].copy()
    k0_row["strike"] = K0
    k0_row["QK"] = (calls.loc[K0, "mid"] + puts.loc[K0, "mid"]) / 2
    k0_row["bid"] = (calls.loc[K0, "bid"] + puts.loc[K0, "bid"]) / 2
    k0_row["ask"] = (calls.loc[K0, "ask"] + puts.loc[K0, "ask"]) / 2
    k0_row["right"] = "K0_AVG"

    selected_options = pd.concat(
        [
            put_otm,
            pd.DataFrame([k0_row]),
            call_otm
        ],
        ignore_index=True
    )

    selected_options = selected_options.sort_values("strike").reset_index(drop=True)

    strikes = selected_options["strike"].astype(float).values
    QK = selected_options["QK"].astype(float).values

    delta_K = np.zeros(len(strikes))

    for i in range(len(strikes)):
        if i == 0:
            delta_K[i] = strikes[i + 1] - strikes[i]
        elif i == len(strikes) - 1:
            delta_K[i] = strikes[i] - strikes[i - 1]
        else:
            delta_K[i] = (strikes[i + 1] - strikes[i - 1]) / 2

    selected_options["delta_K"] = delta_K

    selected_options["contribution"] = (
        selected_options["delta_K"]
        / (selected_options["strike"] ** 2)
        * math.exp(r * T)
        * selected_options["QK"]
    )

    variance = (
        (2 / T) * selected_options["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "F": F,
        "K0": K0,
        "K_star": K_star,
        "T": T,
        "minutes": minutes_to_expiry,
        "num_options": len(selected_options),
        "selected_options": selected_options
    }


def interpolate_to_30d(term_df):
    """
    Interpolate near and next term variances to 30 calendar days.
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target = MINUTES_30_DAYS

    interpolated_variance = (
        T1 * var1 * ((N2 - target) / (N2 - N1))
        + T2 * var2 * ((target - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target)

    return 100 * math.sqrt(interpolated_variance)


def calculate_vix_for_date_v3(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    r=DEFAULT_RISK_FREE_RATE,
    return_raw_options=True
):
    """
    Calculate a replicated 30-day VIX value for one trade date.
    """
    near_exp, next_exp = choose_vix_expirations_v3(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    near_chain = get_chain_at_time(
        root=near_exp["root"],
        expi=near_exp["expiration"],
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    next_chain = get_chain_at_time(
        root=next_exp["root"],
        expi=next_exp["expiration"],
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    near_calc = calc_single_term_variance(
        chain=near_chain,
        minutes_to_expiry=near_exp["minutes"],
        r=r
    )

    next_calc = calc_single_term_variance(
        chain=next_chain,
        minutes_to_expiry=next_exp["minutes"],
        r=r
    )

    term_df = pd.DataFrame([
        {
            "term": "near",
            "root": near_exp["root"],
            "expiration": near_exp["expiration"],
            "minutes": near_exp["minutes"],
            "days": near_exp["days"],
            "variance": near_calc["variance"],
            "F": near_calc["F"],
            "K0": near_calc["K0"],
            "num_options": near_calc["num_options"]
        },
        {
            "term": "next",
            "root": next_exp["root"],
            "expiration": next_exp["expiration"],
            "minutes": next_exp["minutes"],
            "days": next_exp["days"],
            "variance": next_calc["variance"],
            "F": next_calc["F"],
            "K0": next_calc["K0"],
            "num_options": next_calc["num_options"]
        }
    ])

    vix = interpolate_to_30d(term_df)

    result = {
        "trade_date": trade_date,
        "vix": vix,
        "term_df": term_df,
        "near_calc": near_calc,
        "next_calc": next_calc
    }

    if return_raw_options:
        result["near_chain"] = near_chain
        result["next_chain"] = next_chain

    return result

In [8]:
# ============================================================
# Generic tenor interpolation
# ============================================================

def interpolate_to_target_days(term_df, target_days):
    """
    Interpolate near and next term variances to a custom target tenor.

    Example:
        target_days = 30 gives standard 30-day VIX-style interpolation.
        target_days = 9 gives a 9-day VIX-style volatility measure.

    term_df must have exactly two rows:
        near expiration
        next expiration
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target_minutes = target_days * MINUTES_PER_DAY

    if not (N1 <= target_minutes <= N2):
        raise ValueError(
            f"Target tenor {target_days} days is not bracketed by expirations: "
            f"near={N1 / MINUTES_PER_DAY:.2f} days, "
            f"next={N2 / MINUTES_PER_DAY:.2f} days"
        )

    interpolated_variance = (
        T1 * var1 * ((N2 - target_minutes) / (N2 - N1))
        + T2 * var2 * ((target_minutes - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target_minutes)

    return 100 * math.sqrt(interpolated_variance)


def interpolate_to_30d(term_df):
    """
    Backward-compatible wrapper for the old 30-day function.
    """
    return interpolate_to_target_days(term_df, target_days=30)

In [9]:
# ============================================================
# Generic expiration selection for custom target tenors
# ============================================================

def get_friday_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1
):
    """
    Return future Friday SPX/SPXW expirations with days-to-expiry.

    This is the generic expiration universe for custom VIX-style tenors.
    """
    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))

    rows = []

    for exp in all_exps:
        exp_dt = yyyymmdd_to_date(exp)

        # Start with Friday expirations only for methodological consistency.
        if exp_dt.weekday() != 4:
            continue

        root = preferred_root_for_expiration(exp)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms
        )

        if minutes <= min_minutes_to_expiry:
            continue

        rows.append({
            "root": root,
            "expiration": exp,
            "minutes": minutes,
            "days": minutes / MINUTES_PER_DAY
        })

    candidates = pd.DataFrame(rows)

    if candidates.empty:
        raise ValueError(f"No future Friday expirations found for {trade_date}")

    return candidates.sort_values("minutes").reset_index(drop=True)


def choose_expiration_pair_for_target_days(
    trade_date,
    target_days,
    calc_time_ms=CALC_TIME_MS
):
    """
    Choose the two expirations that bracket a target tenor.

    Example:
        target_days = 30 should choose the near and next terms around 30 days.
    """
    candidates = get_friday_expiration_candidates(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    target_minutes = target_days * MINUTES_PER_DAY

    before = candidates[candidates["minutes"] <= target_minutes]
    after = candidates[candidates["minutes"] >= target_minutes]

    if before.empty:
        raise ValueError(
            f"No expiration before target {target_days}d for {trade_date}"
        )

    if after.empty:
        raise ValueError(
            f"No expiration after target {target_days}d for {trade_date}"
        )

    near_idx = before.index[-1]
    next_idx = after.index[0]

    # If target lands exactly on an expiration, use that expiration plus the next one.
    # The interpolation will put 100% weight on the exact expiration.
    if near_idx == next_idx:
        if next_idx + 1 < len(candidates):
            next_idx = next_idx + 1
        elif near_idx - 1 >= 0:
            near_idx = near_idx - 1
        else:
            raise ValueError(f"Could not form expiration pair for {target_days}d")

    near_exp = candidates.loc[near_idx].to_dict()
    next_exp = candidates.loc[next_idx].to_dict()

    return near_exp, next_exp


def get_required_chains_for_target_tenors(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS
):
    """
    For a trade date and a list of target tenors, return:

    1. required_by_tenor:
       one row per target tenor / near-next leg

    2. unique_chains:
       one row per unique chain that needs to be pulled/cached
    """
    rows = []

    for target_days in target_tenor_days:
        near_exp, next_exp = choose_expiration_pair_for_target_days(
            trade_date=trade_date,
            target_days=target_days,
            calc_time_ms=calc_time_ms
        )

        rows.append({
            "target_days": target_days,
            "leg": "near",
            "root": near_exp["root"],
            "expiration": near_exp["expiration"],
            "minutes": near_exp["minutes"],
            "days": near_exp["days"]
        })

        rows.append({
            "target_days": target_days,
            "leg": "next",
            "root": next_exp["root"],
            "expiration": next_exp["expiration"],
            "minutes": next_exp["minutes"],
            "days": next_exp["days"]
        })

    required_by_tenor = pd.DataFrame(rows)

    unique_chains = (
        required_by_tenor[["root", "expiration", "minutes", "days"]]
        .drop_duplicates()
        .sort_values("minutes")
        .reset_index(drop=True)
    )

    return required_by_tenor, unique_chains

In [10]:
# ============================================================
# Multi-tenor VIX-style calculation for one date
# No caching yet. Pulls unique chains in parallel.
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed


def interpolate_variance_to_target_days(term_df, target_days):
    """
    Interpolate near and next term variances to a custom target tenor.

    Returns annualized variance in decimal form.
    Example:
        0.0190 means about 13.78 vol points, because sqrt(0.0190) * 100.
    """
    df = term_df.sort_values("minutes").reset_index(drop=True)

    if len(df) != 2:
        raise ValueError("term_df must have exactly two rows")

    N1 = df.loc[0, "minutes"]
    N2 = df.loc[1, "minutes"]

    T1 = N1 / MINUTES_PER_YEAR
    T2 = N2 / MINUTES_PER_YEAR

    var1 = df.loc[0, "variance"]
    var2 = df.loc[1, "variance"]

    target_minutes = target_days * MINUTES_PER_DAY

    if not (N1 <= target_minutes <= N2):
        raise ValueError(
            f"Target tenor {target_days} days is not bracketed by expirations: "
            f"near={N1 / MINUTES_PER_DAY:.2f} days, "
            f"next={N2 / MINUTES_PER_DAY:.2f} days"
        )

    interpolated_variance = (
        T1 * var1 * ((N2 - target_minutes) / (N2 - N1))
        + T2 * var2 * ((target_minutes - N1) / (N2 - N1))
    ) * (MINUTES_PER_YEAR / target_minutes)

    return interpolated_variance


def interpolate_to_target_days(term_df, target_days):
    """
    Convert interpolated annualized variance to VIX-style volatility points.
    """
    interpolated_variance = interpolate_variance_to_target_days(
        term_df=term_df,
        target_days=target_days
    )

    return 100 * math.sqrt(interpolated_variance)


def pull_unique_chains_parallel(
    trade_date,
    unique_chains,
    calc_time_ms=CALC_TIME_MS,
    max_workers=4
):
    """
    Pull each unique option chain once, in parallel.

    Returns:
        dictionary keyed by (root, expiration)
    """
    chain_results = {}

    def _pull_one_chain(row):
        root = row["root"]
        expiration = int(row["expiration"])

        chain = get_chain_at_time(
            root=root,
            expi=expiration,
            trade_date=trade_date,
            calc_time_ms=calc_time_ms
        )

        return (root, expiration), chain

    rows = [row for _, row in unique_chains.iterrows()]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_pull_one_chain, row)
            for row in rows
        ]

        for future in as_completed(futures):
            key, chain = future.result()
            chain_results[key] = chain

    return chain_results


def calculate_variance_for_unique_chains(
    unique_chains,
    chain_results,
    r
):
    """
    Calculate one VIX-style variance for each unique option expiration.
    """
    rows = []
    calc_results = {}

    for _, row in unique_chains.iterrows():
        root = row["root"]
        expiration = int(row["expiration"])
        minutes = int(row["minutes"])
        days = float(row["days"])

        key = (root, expiration)
        chain = chain_results[key]

        calc = calc_single_term_variance(
            chain=chain,
            minutes_to_expiry=minutes,
            r=r
        )

        calc_results[key] = calc

        rows.append({
            "root": root,
            "expiration": expiration,
            "minutes": minutes,
            "days": days,
            "variance": calc["variance"],
            "F": calc["F"],
            "K0": calc["K0"],
            "K_star": calc["K_star"],
            "num_options": calc["num_options"]
        })

    variance_table = pd.DataFrame(rows).sort_values("minutes").reset_index(drop=True)

    return variance_table, calc_results


def calculate_vix_term_structure_for_date_v5(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol="SOFR",
    max_workers=4,
    return_details=True
):
    """
    Calculate VIX-style implied vol for multiple target tenors on one date.

    Key design:
        - determine all expirations needed
        - pull each unique chain once
        - reuse those chains across all target tenors
    """
    rate = get_interest_rate_for_date_v3(
        symbol=rate_symbol,
        trade_date=trade_date
    )

    required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
        trade_date=trade_date,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms
    )

    chain_results = pull_unique_chains_parallel(
        trade_date=trade_date,
        unique_chains=unique_chains,
        calc_time_ms=calc_time_ms,
        max_workers=max_workers
    )

    variance_table, calc_results = calculate_variance_for_unique_chains(
        unique_chains=unique_chains,
        chain_results=chain_results,
        r=rate
    )

    variance_lookup = {
        (row["root"], int(row["expiration"])): row
        for _, row in variance_table.iterrows()
    }

    output_rows = []

    for target_days in target_tenor_days:
        pair = required_by_tenor[
            required_by_tenor["target_days"] == target_days
        ].copy()

        if len(pair) != 2:
            raise ValueError(f"Expected two expiration rows for target {target_days}d")

        term_rows = []

        for _, leg_row in pair.iterrows():
            key = (leg_row["root"], int(leg_row["expiration"]))
            var_row = variance_lookup[key]

            term_rows.append({
                "term": leg_row["leg"],
                "root": leg_row["root"],
                "expiration": int(leg_row["expiration"]),
                "minutes": int(leg_row["minutes"]),
                "days": float(leg_row["days"]),
                "variance": float(var_row["variance"]),
                "F": float(var_row["F"]),
                "K0": float(var_row["K0"]),
                "num_options": int(var_row["num_options"])
            })

        term_df = pd.DataFrame(term_rows).sort_values("minutes").reset_index(drop=True)

        implied_variance = interpolate_variance_to_target_days(
            term_df=term_df,
            target_days=target_days
        )

        implied_vol = 100 * math.sqrt(implied_variance)

        output_rows.append({
            "trade_date": trade_date,
            "target_days": target_days,
            "rate_symbol": rate_symbol,
            "rate_pct": rate * 100,
            "implied_variance": implied_variance,
            "vix_style_vol": implied_vol,
            "near_root": term_df.loc[0, "root"],
            "near_expiration": term_df.loc[0, "expiration"],
            "near_days": term_df.loc[0, "days"],
            "near_variance": term_df.loc[0, "variance"],
            "next_root": term_df.loc[1, "root"],
            "next_expiration": term_df.loc[1, "expiration"],
            "next_days": term_df.loc[1, "days"],
            "next_variance": term_df.loc[1, "variance"]
        })

    result_df = pd.DataFrame(output_rows).sort_values("target_days").reset_index(drop=True)

    result = {
        "trade_date": trade_date,
        "rate_symbol": rate_symbol,
        "rate_decimal": rate,
        "rate_pct": rate * 100,
        "results_df": result_df
    }

    if return_details:
        result["required_by_tenor"] = required_by_tenor
        result["unique_chains"] = unique_chains
        result["variance_table"] = variance_table
        result["chain_results"] = chain_results
        result["calc_results"] = calc_results

    return result

In [11]:
# ============================================================
# Raw option-chain cache
# ============================================================

from pathlib import Path

CHAIN_CACHE_DIR = Path("data/raw/thetadata_chains")
CHAIN_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_time_label(calc_time_ms):
    """
    Convert calc time to compact label.

    Example:
        57600000 -> '160000'
    """
    time_str = ms_to_time_string(calc_time_ms)
    return time_str.replace(":", "").replace(".000", "")


def get_chain_cache_path(root, trade_date, expi, calc_time_ms):
    """
    Local cache file path for one option chain.
    """
    time_label = cache_time_label(calc_time_ms)

    filename = f"{root}_{int(trade_date)}_{int(expi)}_{time_label}.pkl"

    return CHAIN_CACHE_DIR / filename


def get_chain_at_time_cached(
    root,
    expi,
    trade_date,
    calc_time_ms,
    force_refresh=False,
    verbose=False
):
    """
    Cached wrapper around get_chain_at_time().

    If the chain exists locally, load it.
    Otherwise pull from ThetaData and save it.
    """

    cache_path = get_chain_cache_path(
        root=root,
        trade_date=trade_date,
        expi=expi,
        calc_time_ms=calc_time_ms
    )

    if cache_path.exists() and not force_refresh:
        if verbose:
            print(f"Loading from cache: {cache_path}")

        return pd.read_pickle(cache_path)

    if verbose:
        print(f"Pulling from ThetaData: {root} {trade_date} {expi}")

    chain = get_chain_at_time(
        root=root,
        expi=expi,
        trade_date=trade_date,
        calc_time_ms=calc_time_ms
    )

    chain.to_pickle(cache_path)

    if verbose:
        print(f"Saved to cache: {cache_path}")

    return chain

In [12]:
# ============================================================
# Cached parallel chain pulling for multi-tenor engine
# ============================================================

def pull_unique_chains_parallel_cached(
    trade_date,
    unique_chains,
    calc_time_ms=CALC_TIME_MS,
    max_workers=4,
    force_refresh=False,
    verbose=False
):
    """
    Pull/load each unique option chain once, in parallel.

    Uses local cache:
        if cached file exists -> load it
        else -> pull from ThetaData and save it
    """
    chain_results = {}

    def _pull_one_chain(row):
        root = row["root"]
        expiration = int(row["expiration"])

        chain = get_chain_at_time_cached(
            root=root,
            expi=expiration,
            trade_date=trade_date,
            calc_time_ms=calc_time_ms,
            force_refresh=force_refresh,
            verbose=verbose
        )

        return (root, expiration), chain

    rows = [row for _, row in unique_chains.iterrows()]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_pull_one_chain, row)
            for row in rows
        ]

        for future in as_completed(futures):
            key, chain = future.result()
            chain_results[key] = chain

    return chain_results


In [13]:
# ============================================================
# Multi-tenor VIX-style calculation using raw chain cache
# ============================================================

def calculate_vix_term_structure_for_date_v5_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol="SOFR",
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False
):
    """
    Calculate VIX-style implied vol for multiple target tenors on one date.

    Uses:
        - SOFR rate
        - Friday SPX/SPXW expirations
        - parallel chain loading
        - local raw chain cache

    Key design:
        - determine all expirations needed
        - load/pull each unique chain once
        - reuse those chains across all target tenors
    """
    rate = get_interest_rate_for_date_v3(
        symbol=rate_symbol,
        trade_date=trade_date
    )

    required_by_tenor, unique_chains = get_required_chains_for_target_tenors(
        trade_date=trade_date,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms
    )

    chain_results = pull_unique_chains_parallel_cached(
        trade_date=trade_date,
        unique_chains=unique_chains,
        calc_time_ms=calc_time_ms,
        max_workers=max_workers,
        force_refresh=force_refresh,
        verbose=verbose
    )

    variance_table, calc_results = calculate_variance_for_unique_chains(
        unique_chains=unique_chains,
        chain_results=chain_results,
        r=rate
    )

    variance_lookup = {
        (row["root"], int(row["expiration"])): row
        for _, row in variance_table.iterrows()
    }

    output_rows = []

    for target_days in target_tenor_days:
        pair = required_by_tenor[
            required_by_tenor["target_days"] == target_days
        ].copy()

        if len(pair) != 2:
            raise ValueError(f"Expected two expiration rows for target {target_days}d")

        term_rows = []

        for _, leg_row in pair.iterrows():
            key = (leg_row["root"], int(leg_row["expiration"]))
            var_row = variance_lookup[key]

            term_rows.append({
                "term": leg_row["leg"],
                "root": leg_row["root"],
                "expiration": int(leg_row["expiration"]),
                "minutes": int(leg_row["minutes"]),
                "days": float(leg_row["days"]),
                "variance": float(var_row["variance"]),
                "F": float(var_row["F"]),
                "K0": float(var_row["K0"]),
                "num_options": int(var_row["num_options"])
            })

        term_df = pd.DataFrame(term_rows).sort_values("minutes").reset_index(drop=True)

        implied_variance = interpolate_variance_to_target_days(
            term_df=term_df,
            target_days=target_days
        )

        implied_vol = 100 * math.sqrt(implied_variance)

        output_rows.append({
            "trade_date": trade_date,
            "target_days": target_days,
            "rate_symbol": rate_symbol,
            "rate_pct": rate * 100,
            "implied_variance": implied_variance,
            "vix_style_vol": implied_vol,
            "near_root": term_df.loc[0, "root"],
            "near_expiration": term_df.loc[0, "expiration"],
            "near_days": term_df.loc[0, "days"],
            "near_variance": term_df.loc[0, "variance"],
            "next_root": term_df.loc[1, "root"],
            "next_expiration": term_df.loc[1, "expiration"],
            "next_days": term_df.loc[1, "days"],
            "next_variance": term_df.loc[1, "variance"]
        })

    result_df = pd.DataFrame(output_rows).sort_values("target_days").reset_index(drop=True)

    result = {
        "trade_date": trade_date,
        "rate_symbol": rate_symbol,
        "rate_decimal": rate,
        "rate_pct": rate * 100,
        "results_df": result_df
    }

    if return_details:
        result["required_by_tenor"] = required_by_tenor
        result["unique_chains"] = unique_chains
        result["variance_table"] = variance_table
        result["chain_results"] = chain_results
        result["calc_results"] = calc_results

    return result


## v0.6 methodology layer

The base v0.5 engine remains intact. The cells below add:

1. A ThetaData-derived SPX trading calendar.
2. A holiday-adjusted Friday-cycle expiration universe.
3. A v0.6 wrapper around the cached term-structure engine.
4. Processed-history batch/update helpers using the v0.6 methodology key.


In [14]:

# ============================================================
# v0.6 processed data settings
# ============================================================

PROCESSED_DATA_DIR = Path("data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TERM_STRUCTURE_HISTORY_CSV = PROCESSED_DATA_DIR / "vix_term_structure_history.csv"
TERM_STRUCTURE_HISTORY_PARQUET = PROCESSED_DATA_DIR / "vix_term_structure_history.parquet"

SPX_TRADING_DATES_CSV = PROCESSED_DATA_DIR / "spx_trading_dates.csv"

METHODOLOGY_VERSION = "v0.6_friday_cycle_holiday_adjusted_sofr_1600"
EXPIRATION_UNIVERSE = "friday_cycle_holiday_adjusted"
QUOTE_TIME_LABEL = cache_time_label(CALC_TIME_MS)


In [15]:

# ============================================================
# SPX trading calendar from ThetaData index EOD history
# ============================================================

from datetime import timedelta


def get_spx_index_eod_history_v3(start_date, end_date, verbose=False):
    """
    Pull SPX index EOD history from ThetaData v3.

    Used primarily to identify actual SPX trading dates.
    """
    url = BASE_URL_V3 + "/index/history/eod"

    params = {
        "symbol": "SPX",
        "start_date": int(start_date),
        "end_date": int(end_date),
        "format": "json",
    }

    r = requests.get(url, params=params, timeout=60)

    if verbose:
        print("URL:", r.url)
        print("Status code:", r.status_code)
        print(r.text[:1000])

    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data)

    if df.empty:
        raise ValueError(f"No SPX index EOD data returned for {start_date} to {end_date}")

    if "last_trade" not in df.columns:
        raise ValueError(f"Expected 'last_trade' column. Columns: {list(df.columns)}")

    df["trade_date"] = (
        pd.to_datetime(df["last_trade"])
        .dt.strftime("%Y%m%d")
        .astype(int)
    )

    df = df.sort_values("trade_date").reset_index(drop=True)

    return df


def update_spx_trading_dates_file(start_date, end_date, force_refresh=False):
    """
    Pull actual SPX trading dates from ThetaData and save/update local CSV.
    """
    if SPX_TRADING_DATES_CSV.exists() and not force_refresh:
        existing = pd.read_csv(SPX_TRADING_DATES_CSV)
    else:
        existing = pd.DataFrame(columns=["trade_date"])

    eod_df = get_spx_index_eod_history_v3(
        start_date=start_date,
        end_date=end_date,
        verbose=False,
    )

    new_dates = eod_df[["trade_date"]].drop_duplicates().copy()

    combined = (
        pd.concat([existing[["trade_date"]], new_dates], ignore_index=True)
        .drop_duplicates()
        .sort_values("trade_date")
        .reset_index(drop=True)
    )

    combined.to_csv(SPX_TRADING_DATES_CSV, index=False)

    print(f"Saved SPX trading dates to: {SPX_TRADING_DATES_CSV}")
    print("Rows:", len(combined))
    print("Min date:", combined["trade_date"].min())
    print("Max date:", combined["trade_date"].max())

    return combined


def load_spx_trading_dates():
    """
    Load locally saved SPX trading dates.
    """
    if not SPX_TRADING_DATES_CSV.exists():
        raise FileNotFoundError(
            f"{SPX_TRADING_DATES_CSV} does not exist yet. "
            "Run update_spx_trading_dates_file first."
        )

    df = pd.read_csv(SPX_TRADING_DATES_CSV)
    return sorted(df["trade_date"].astype(int).unique().tolist())


In [23]:
# ============================================================
# v0.6 expiration universe:
# Friday-cycle, holiday-adjusted
# ============================================================

METHODOLOGY_VERSION = "v0.6_friday_cycle_holiday_adjusted_sofr_1600"
EXPIRATION_UNIVERSE = "friday_cycle_holiday_adjusted"
QUOTE_TIME_LABEL = cache_time_label(CALC_TIME_MS)


def int_date_from_date(dt):
    """
    Convert datetime.date to YYYYMMDD integer.
    """
    return int(dt.strftime("%Y%m%d"))


def next_calendar_friday_after_date(dt):
    """
    Return the next calendar Friday after dt, including the same week.
    Used for detecting Friday-cycle expirations shifted earlier by a Friday holiday.
    """
    days_until_friday = (4 - dt.weekday()) % 7

    if days_until_friday == 0:
        return dt

    return dt + timedelta(days=days_until_friday)


def is_last_trading_day_before_closed_friday(exp_yyyymmdd, spx_trading_dates):
    """
    True if exp_yyyymmdd is the last SPX trading day before a closed calendar Friday.

    This handles cases like:
        - Good Friday
        - July 4 on a Friday
        - Christmas on a Friday
        - Any other Friday market holiday

    It is more general than simply checking for Thursday.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)
    exp_int = int(exp_yyyymmdd)

    trading_date_set = set(int(x) for x in spx_trading_dates)

    # The shifted expiration itself must be an SPX trading day.
    if exp_int not in trading_date_set:
        return False

    # Normal Friday expirations are handled separately.
    if exp_dt.weekday() == 4:
        return False

    next_friday = next_calendar_friday_after_date(exp_dt)
    next_friday_int = int_date_from_date(next_friday)

    # If the Friday is open, this is not a holiday-adjusted Friday-cycle expiry.
    if next_friday_int in trading_date_set:
        return False

    # There must be no trading day between the expiration date and the closed Friday.
    # That makes exp_yyyymmdd the final trading day before the Friday holiday.
    check_dates = pd.date_range(
        start=exp_dt + timedelta(days=1),
        end=next_friday - timedelta(days=1),
        freq="D"
    )

    for d in check_dates:
        d_int = int(d.strftime("%Y%m%d"))

        if d_int in trading_date_set:
            return False

    return True


def is_friday_cycle_expiration_v6(exp_yyyymmdd, spx_trading_dates):
    """
    Friday-cycle expiration definition:

    1. Normal Friday expirations.
    2. Non-Friday expirations that are the final SPX trading day
       before a closed calendar Friday.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if exp_dt.weekday() == 4:
        return True

    if is_last_trading_day_before_closed_friday(exp_yyyymmdd, spx_trading_dates):
        return True

    return False


def is_holiday_adjusted_monthly_expiration_v6(exp_yyyymmdd, spx_trading_dates):
    """
    True if this expiration is the holiday-adjusted version of a standard
    monthly third-Friday SPX expiration.
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    if exp_dt.weekday() == 4:
        return False

    next_friday = next_calendar_friday_after_date(exp_dt)

    if not is_third_friday(next_friday):
        return False

    return is_last_trading_day_before_closed_friday(
        exp_yyyymmdd=exp_yyyymmdd,
        spx_trading_dates=spx_trading_dates
    )


def preferred_root_for_expiration_v6(exp_yyyymmdd, spx_trading_dates):
    """
    Prefer SPX for standard monthly AM-settled expirations.

    Includes:
        - normal third-Friday SPX expirations
        - holiday-adjusted monthly SPX expirations shifted earlier
    """
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    # Normal monthly AM-settled SPX.
    if is_third_friday(exp_dt) and exp_yyyymmdd in spx_exps:
        return "SPX"

    # Holiday-adjusted monthly AM-settled SPX.
    if (
        is_holiday_adjusted_monthly_expiration_v6(exp_yyyymmdd, spx_trading_dates)
        and exp_yyyymmdd in spx_exps
    ):
        return "SPX"

    # Otherwise prefer SPXW weekly/daily PM-settled options.
    if exp_yyyymmdd in spxw_exps:
        return "SPXW"

    if exp_yyyymmdd in spx_exps:
        return "SPX"

    raise ValueError(f"Expiration {exp_yyyymmdd} not found in SPX or SPXW expiration lists")


def get_friday_cycle_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1,
    spx_trading_dates=None
):
    """
    Return future Friday-cycle SPX/SPXW expirations with days-to-expiry.

    Includes:
        - normal Friday expirations
        - holiday-adjusted expirations shifted earlier when Friday is closed
    """
    if spx_trading_dates is None:
        spx_trading_dates = load_spx_trading_dates()

    all_exps = sorted(set(spx_exps).union(set(spxw_exps)))

    rows = []

    for exp in all_exps:
        if not is_friday_cycle_expiration_v6(exp, spx_trading_dates):
            continue

        root = preferred_root_for_expiration_v6(exp, spx_trading_dates)

        minutes = minutes_to_expiry_vix_method(
            trade_date=trade_date,
            exp_yyyymmdd=exp,
            root=root,
            calc_time_ms=calc_time_ms
        )

        if minutes <= min_minutes_to_expiry:
            continue

        rows.append({
            "root": root,
            "expiration": exp,
            "minutes": minutes,
            "days": minutes / MINUTES_PER_DAY
        })

    candidates = pd.DataFrame(rows)

    if candidates.empty:
        raise ValueError(f"No future Friday-cycle expirations found for {trade_date}")

    return candidates.sort_values("minutes").reset_index(drop=True)


# Redefine the existing function name so downstream v0.5-style code
# now uses the corrected v0.6 expiration universe.
def get_friday_expiration_candidates(
    trade_date,
    calc_time_ms=CALC_TIME_MS,
    min_minutes_to_expiry=1
):
    """
    Backward-compatible function name.

    In v0.6, this means Friday-cycle, holiday-adjusted candidates.
    """
    return get_friday_cycle_expiration_candidates(
        trade_date=trade_date,
        calc_time_ms=calc_time_ms,
        min_minutes_to_expiry=min_minutes_to_expiry,
        spx_trading_dates=load_spx_trading_dates()
    )

In [17]:

# ============================================================
# v0.6 cached calculation wrapper
# ============================================================


def calculate_vix_term_structure_for_date_v6_cached(
    trade_date,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol="SOFR",
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
):
    """
    v0.6 wrapper around the cached multi-tenor engine.

    The underlying engine is the same as v0.5, but the expiration universe has been
    redefined above as Friday-cycle, holiday-adjusted.
    """
    return calculate_vix_term_structure_for_date_v5_cached(
        trade_date=trade_date,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        return_details=return_details,
        verbose=verbose,
    )


In [18]:

# ============================================================
# v0.6 processed history load / save / update functions
# ============================================================


def load_existing_term_structure_history():
    """
    Load existing processed history if it exists.

    Prefer parquet if available, otherwise CSV.
    """
    if TERM_STRUCTURE_HISTORY_PARQUET.exists():
        try:
            return pd.read_parquet(TERM_STRUCTURE_HISTORY_PARQUET)
        except Exception as e:
            print(f"Could not read parquet file. Falling back to CSV if available. Error: {e}")

    if TERM_STRUCTURE_HISTORY_CSV.exists():
        return pd.read_csv(TERM_STRUCTURE_HISTORY_CSV)

    return pd.DataFrame()


def save_term_structure_history(history_df):
    """
    Save processed history.

    Always writes CSV.
    Tries to write parquet if the environment supports it.
    """
    history_df = history_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)

    history_df.to_csv(TERM_STRUCTURE_HISTORY_CSV, index=False)

    try:
        history_df.to_parquet(TERM_STRUCTURE_HISTORY_PARQUET, index=False)
        print(f"Saved CSV: {TERM_STRUCTURE_HISTORY_CSV}")
        print(f"Saved parquet: {TERM_STRUCTURE_HISTORY_PARQUET}")
    except Exception as e:
        print(f"Saved CSV: {TERM_STRUCTURE_HISTORY_CSV}")
        print(f"Could not save parquet. CSV is still saved. Error: {e}")


def upsert_term_structure_history(new_rows_df):
    """
    Append/replace processed history rows.

    Replacement key:
        trade_date + target_days + methodology_version + quote_time
    """
    existing_df = load_existing_term_structure_history()

    if new_rows_df.empty:
        print("No new rows to save.")
        return existing_df

    key_cols = [
        "trade_date",
        "target_days",
        "methodology_version",
        "quote_time",
    ]

    if existing_df.empty:
        updated_df = new_rows_df.copy()
    else:
        existing_key = existing_df[key_cols].astype(str).agg("|".join, axis=1)
        new_key = new_rows_df[key_cols].astype(str).agg("|".join, axis=1)

        existing_without_replaced = existing_df[~existing_key.isin(set(new_key))].copy()

        updated_df = pd.concat(
            [existing_without_replaced, new_rows_df],
            ignore_index=True,
        )

    updated_df = updated_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)

    save_term_structure_history(updated_df)

    print("Rows in updated history:", len(updated_df))
    print("Min date:", updated_df["trade_date"].min())
    print("Max date:", updated_df["trade_date"].max())

    return updated_df


In [19]:

# ============================================================
# v0.6 batch calculation and missing-date update functions
# ============================================================


def run_vix_term_structure_batch_v6(
    trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol="SOFR",
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Calculate v0.6 multi-tenor VIX-style term structure for a list of trade dates.

    Returns:
        results_df: one row per trade_date / target_days
        errors_df: one row per failed trade_date, if any
    """
    all_results = []
    errors = []

    for i, trade_date in enumerate(trade_dates, start=1):
        print(f"[{i}/{len(trade_dates)}] Calculating {trade_date}...")

        try:
            result = calculate_vix_term_structure_for_date_v6_cached(
                trade_date=trade_date,
                target_tenor_days=target_tenor_days,
                calc_time_ms=calc_time_ms,
                rate_symbol=rate_symbol,
                max_workers=max_workers,
                force_refresh=force_refresh,
                return_details=False,
                verbose=False,
            )

            df = result["results_df"].copy()

            df["methodology_version"] = METHODOLOGY_VERSION
            df["expiration_universe"] = EXPIRATION_UNIVERSE
            df["quote_time"] = QUOTE_TIME_LABEL
            df["calc_time_ms"] = calc_time_ms

            all_results.append(df)

        except Exception as e:
            error_row = {
                "trade_date": trade_date,
                "error": str(e),
            }

            errors.append(error_row)
            print(f"ERROR on {trade_date}: {e}")

            if not continue_on_error:
                raise

    if len(all_results) > 0:
        results_df = pd.concat(all_results, ignore_index=True)
        results_df = results_df.sort_values(["trade_date", "target_days"]).reset_index(drop=True)
    else:
        results_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    return results_df, errors_df


def get_completed_trade_dates_v6(history_df=None, target_tenor_days=TARGET_TENOR_DAYS):
    """
    Return trade dates that already have a complete set of target tenors
    for the v0.6 methodology version and quote time.
    """
    if history_df is None:
        history_df = load_existing_term_structure_history()

    if history_df.empty:
        return set()

    filtered = history_df[
        (history_df["methodology_version"] == METHODOLOGY_VERSION)
        & (history_df["quote_time"].astype(str) == str(QUOTE_TIME_LABEL))
    ].copy()

    if filtered.empty:
        return set()

    target_set = set(int(x) for x in target_tenor_days)
    completed_dates = set()

    for trade_date, group in filtered.groupby("trade_date"):
        tenor_set = set(int(x) for x in group["target_days"].unique())

        if target_set.issubset(tenor_set):
            completed_dates.add(int(trade_date))

    return completed_dates


def find_missing_trade_dates_v6(candidate_trade_dates, target_tenor_days=TARGET_TENOR_DAYS):
    """
    Given candidate trade dates, return only dates that do not yet have
    a complete v0.6 processed history.
    """
    history_df = load_existing_term_structure_history()

    completed_dates = get_completed_trade_dates_v6(
        history_df=history_df,
        target_tenor_days=target_tenor_days,
    )

    candidate_dates_clean = sorted(set(int(x) for x in candidate_trade_dates))

    missing_dates = [
        d for d in candidate_dates_clean
        if d not in completed_dates
    ]

    return missing_dates


def update_term_structure_history_for_dates_v6(
    candidate_trade_dates,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol="SOFR",
    max_workers=4,
    force_refresh=False,
    continue_on_error=True,
):
    """
    Update processed v0.6 term-structure history for missing dates only.
    """
    missing_dates = find_missing_trade_dates_v6(
        candidate_trade_dates=candidate_trade_dates,
        target_tenor_days=target_tenor_days,
    )

    print("Candidate dates:", len(set(candidate_trade_dates)))
    print("Missing/incomplete v0.6 dates:", len(missing_dates))

    if len(missing_dates) == 0:
        print("v0.6 history is already complete for the supplied dates.")
        return load_existing_term_structure_history(), pd.DataFrame()

    results_df, errors_df = run_vix_term_structure_batch_v6(
        trade_dates=missing_dates,
        target_tenor_days=target_tenor_days,
        calc_time_ms=calc_time_ms,
        rate_symbol=rate_symbol,
        max_workers=max_workers,
        force_refresh=force_refresh,
        continue_on_error=continue_on_error,
    )

    if not results_df.empty:
        updated_history_df = upsert_term_structure_history(results_df)
    else:
        updated_history_df = load_existing_term_structure_history()

    return updated_history_df, errors_df



## v0.6 validation cells

Run these after `Kernel → Restart Kernel and Run All Cells` reaches this section. The first validation builds a small SPX trading calendar around Good Friday 2024, where Friday 2024-03-29 was not a trading day. The expiration universe should include 2024-03-28 as a holiday-adjusted Thursday expiration.


In [24]:
# ============================================================
# Build broad SPX trading calendar for v0.6 validation
# ============================================================

spx_trading_dates_validation_df = update_spx_trading_dates_file(
    start_date=20240101,
    end_date=20250930,
    force_refresh=True
)

spx_trading_dates_validation = load_spx_trading_dates()

print("Trading dates loaded:", len(spx_trading_dates_validation))
print("First:", min(spx_trading_dates_validation))
print("Last:", max(spx_trading_dates_validation))

print("Good Friday 2024 checks:")
print("20240328 in calendar:", 20240328 in spx_trading_dates_validation)
print("20240329 in calendar:", 20240329 in spx_trading_dates_validation)
print("20240401 in calendar:", 20240401 in spx_trading_dates_validation)

print()
print("July 4 2025 checks:")
print("20250703 in calendar:", 20250703 in spx_trading_dates_validation)
print("20250704 in calendar:", 20250704 in spx_trading_dates_validation)
print("20250707 in calendar:", 20250707 in spx_trading_dates_validation)

HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:25503/v3/index/history/eod?symbol=SPX&start_date=20240101&end_date=20250930&format=json

In [22]:

# ============================================================
# v0.6 one-date sanity check
# ============================================================

term_structure_20240315_v6 = calculate_vix_term_structure_for_date_v6_cached(
    trade_date=20240315,
    target_tenor_days=TARGET_TENOR_DAYS,
    calc_time_ms=CALC_TIME_MS,
    rate_symbol="SOFR",
    max_workers=4,
    force_refresh=False,
    return_details=True,
    verbose=False,
)

display(term_structure_20240315_v6["results_df"])
print("Unique chains needed:", len(term_structure_20240315_v6["unique_chains"]))


,trade_date,target_days,rate_symbol,rate_pct,implied_variance,vix_style_vol,near_root,near_expiration,near_days,near_variance,next_root,next_expiration,next_days,next_variance
0,20240315,9,SOFR,5.31,0.020391,14.279555,SPXW,20240322,7.0,0.020777,SPXW,20240328,13.0,0.019974
1,20240315,12,SOFR,5.31,0.020052,14.160512,SPXW,20240322,7.0,0.020777,SPXW,20240328,13.0,0.019974
2,20240315,15,SOFR,5.31,0.019918,14.113288,SPXW,20240328,13.0,0.019974,SPXW,20240405,21.0,0.019816
3,20240315,18,SOFR,5.31,0.019858,14.092013,SPXW,20240328,13.0,0.019974,SPXW,20240405,21.0,0.019816
4,20240315,21,SOFR,5.31,0.019816,14.076797,SPXW,20240405,21.0,0.019816,SPXW,20240411,27.0,0.020879
5,20240315,24,SOFR,5.31,0.020414,14.287631,SPXW,20240405,21.0,0.019816,SPXW,20240411,27.0,0.020879
6,20240315,27,SOFR,5.31,0.020879,14.449487,SPXW,20240411,27.0,0.020879,SPXW,20240412,28.0,0.021019
7,20240315,30,SOFR,5.31,0.021000,14.491333,SPXW,20240412,28.0,0.021019,SPXW,20240418,34.0,0.020968
8,20240315,33,SOFR,5.31,0.020976,14.482937,SPXW,20240412,28.0,0.021019,SPXW,20240418,34.0,0.020968


Unique chains needed: 6
